# Day 3 — Data cleaning

Standalone, reusable cleaning pipeline for the Disney+ titles dataset. Later stages (Day 4 EDA, model training, etc.) should import `clean_data()` rather than re-running exploratory code — this notebook's job is turning the raw CSV into a clean one.

**Note on this conversion:** the original `clean_data.py` exposes a command-line entry point (`argparse`, `--input`/`--output` flags) for running it from a terminal. That doesn't translate directly into a notebook — a notebook's kernel passes its own arguments (like `-f <connection_file>`) that `argparse` isn't expecting, so calling `main()` unmodified here would crash. The cell below calls `clean_data()` directly with the same default paths instead. If you need the CLI version, `clean_data.py` still has it.

In [ ]:
import pandas as pd
from pathlib import Path

# Raw data lives at the project root, one level up from stage01/
# (confirmed from the actual repo layout -- adjust here if that changes).
# The cleaned output stays inside stage01/, since it's this stage's own
# artifact -- stage02 can reference it via a relative path from there.
SCRIPT_DIR = Path.cwd()
RAW_PATH = SCRIPT_DIR.parent / "disney_plus_titles.csv"
CLEAN_PATH = SCRIPT_DIR / "disney_plus_titles_clean.csv"

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the raw Disney+ titles dataframe.

    Decisions made here (full "why" for each lives in workflow.md):
      1. Drop rows with missing `rating` -- that's the prediction target;
         a row with no label can't be trained on or scored against.
      2. Leave `director` / `cast` columns in place but don't engineer
         features from them (33% / 13% missing, high-cardinality).
      3. Fill missing `country` with "Unknown" rather than dropping rows
         -- country is a feature we want to keep.
      4. Split `duration` into `duration_value` (int) + `duration_unit`
         (min/Season) -- the raw column mixes two different units
         depending on `type`.
      5. Split `listed_in` into `genre_list` -- one title can carry
         multiple genres.
      6. Parse `date_added` to a real datetime, where present.

    Known limitation, not fixed here: `duration_unit` perfectly predicts
    `type` (every Movie is "min", every TV Show is "Season"). If a later
    stage predicts `type` instead of `rating`, drop `duration` from the
    feature set first -- otherwise the model is just reading a
    disguised copy of the label.
    """
    df = df.copy()
    start_rows = len(df)

    # duplicates
    dupes = df.duplicated().sum()
    df = df.drop_duplicates()

    # missing values
    df = df.dropna(subset=["rating"])
    df["country"] = df["country"].fillna("Unknown")
    # date_added's 3 missing values are left as-is (not used as a
    # feature this week) rather than inventing a fake date.

    # mismatched types
    duration_split = df["duration"].str.extract(
        r"(?P<duration_value>\d+)\s*(?P<duration_unit>\w+)"
    )
    df["duration_value"] = pd.to_numeric(duration_split["duration_value"])
    df["duration_unit"] = duration_split["duration_unit"].replace(
        {"Season": "Season", "Seasons": "Season"}  # normalize singular/plural
    )
    df["genre_list"] = df["listed_in"].str.split(", ")
    df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")

    dropped = start_rows - len(df)
    print(f"Duplicate rows found: {dupes}")
    print(f"Rows: {start_rows} -> {len(df)} ({dropped} dropped, missing `rating`)")
    return df

## Run the cleaning pipeline

In [ ]:
raw_df = pd.read_csv(RAW_PATH)
clean_df = clean_data(raw_df)
clean_df.to_csv(CLEAN_PATH, index=False)
print(f"Saved -> {CLEAN_PATH} ({len(clean_df)} rows)")
clean_df.head(3)